In [0]:
%python
# generator 
def fun ():
    print("The Gegerator  is  generatin  the  value ")
    yield "x"
    yield "y"
    yield "z"
    print("The Gegerator  is  generatin  the  value ")

gen =  fun()
for i  in  gen:
    print(i)




## Cell 1 — Basic Generators: Why, What, How

**Why?** A generator produces values *lazily* — one at a time, on demand — instead of building the whole collection in memory. For large or streamed data this is the difference between O(n) memory and O(1) memory per item, and it lets you model a sequence you may not even know the end of.

**What?** Any function containing `yield` is a generator *function*. Calling it runs **zero** lines of the body — it returns a generator *object* that holds the paused state. Each `yield` both emits a value to the consumer and freezes execution exactly where it is.

**How?** The `for` loop repeatedly calls `next()` under the hood: the function resumes after the paused `yield`, runs until the next `yield` (here emitting `"x"`, `"y"`, `"z"`), and when the body ends it raises `StopIteration`, which the loop catches silently. The two `print()` calls reveal the asymmetry — the first runs before `"x"` is delivered; the second only fires on the final `next()`, when the loop discovers the function is exhausted.

In [0]:
%python
#Infinite generator
def  fun ():
    count = 1 
    while True :
        yield f"the value of  the  count is {count}"
        count+=1 
in_gen1 =  fun()
in_gen2 =  fun()
for _ in range(7):
    print( next(in_gen1 ,  "the value is  not  available"))
for _ in range(8):
    print( next(in_gen2 , "The  value  is  not  available"))




## Cell 2 — Infinite Generators: Why, What, How

**Why?** With lazy evaluation, "infinite" sequences are free: a counter, a stream of IDs, a paginated API cursor. Memory stays constant because only the current state (`count`) is ever stored — the sequence itself is never materialized.

**What?** `while True` means the generator never raises `StopIteration`; it can produce values forever. `in_gen1` and `in_gen2` are **independent instances** of the same function — each keeps its own `count`, which is why both start at 1. Generator state lives in the *object*, not the function definition.

**How?** `next(gen, default)` is the safety valve: when a generator *is* exhausted it returns the default instead of raising `StopIteration`. Here it never triggers (the loop cannot end) — but it is the pattern you want whenever a source may legitimately run dry mid-consumption. Pulling values on demand, bounded by `range(7)` / `range(8)` rather than by the source, is exactly how you consume an infinite stream: the *consumer* decides when to stop.

In [0]:
%python
# sendig  the value to the   to  generator  

def fun():
    print("Hello for the funtiom")
    order =  yield    #  this will require   the second  call  for   funtion  ... 
    while(1):
        print(f"the order is {order}")
        order= yield   #  this is  for  the    to  stop the  the infinite  loop  : 

gen1=  fun()
gen2=fun()
next(gen1)
gen1.send(2000)
next(gen2)
gen2.send(3000)
gen2.send(30000)

## Cell 3 — `send()`: Two-Way Generators: Why, What, How

**Why?** A plain generator only *emits*. `send()` turns `yield` into a two-way channel — the consumer can push data *into* the paused function. This is the stepping stone to coroutines, data pipelines, and (historically) `async`/`await`.

**What?** Inside the function, `order = yield` is a full expression: `yield` pauses execution, and when execution resumes, the sent value becomes `order`. The first `yield` only *receives* — there is nothing to give back yet — so it must be crossed once before any real data can flow in.

**How?** A fresh generator is paused at the *top of the body*, before any `yield`. Calling `send(non-None)` on a just-started generator raises `TypeError: can't send non-None value to a just-started generator`. The fix is **priming**: call `next(gen)` first to advance to the first `yield`, then `send(2000)` resumes it with `2000` delivered as the value of that `yield`. Inside `while(1)`, each subsequent `yield` becomes the pause point where the next `send()` lands — so the final `gen2.send(30000)` simply resumes at the second `order = yield` and prints one more line. Rule of thumb: **always prime, then send**.

In [0]:
%python  
 
def local_chai():
    yield "Masala Chai"
    yield "Green  Tea"
    yield "Black  Tea"
def imported_chai():
    yield"Matcha"
    yield "olang"
def full_menu():
    yield from local_chai()
    yield from imported_chai()
for  chai  in  full_menu():
    print(chai)    #  this  for  loop  give all  the thing


# as the  above logic  give all the  chai  what  if  i want to  get  the  chai  one by one  then  we  can  use  the  next or  close  

def chai_stall():
    try:
        while(1):
            order=yield "waiting for chai order"
    except:
        print("stall closed")

            
stall = chai_stall()
print(next(stall))
stall.close()  # clean up  
        

    


## `yield from` & `close()` — Simple Notes

### `yield from` — What is it?
* It **connects one generator to another generator** (called a sub-generator).
* `yield from local_chai()` simply means: *"whatever `local_chai()` gives, pass it directly to my consumer — I won't touch it."*
* So the consumer gets **one simple flat sequence**, without knowing it actually came from two different generators.

### Why do we use it?
* To **join** many small generators into one — like combining local + imported chai into one menu.
* It is not just a shortcut for values — it also passes `send()` and `throw()` through to the sub-generator.

### `close()` — What is it?
* It is the polite way to **shut down** a generator from outside.
* Internally it raises **`GeneratorExit`** inside the generator at its paused `yield` — the generator can catch it and do cleanup (close a file, a connection... a stall).
* After `close()`, the generator is **dead** — calling `next()` on it gives `StopIteration`.

### Easy way to remember
* `yield from` = **join** generators (later became the idea behind `await` in async)
* `close()` = **stop** a generator cleanly, with a chance to clean up

### Interview one-liner
*"`yield from` delegates to a sub-generator — passing values both ways — and `close()` shuts a generator down by raising `GeneratorExit` at its paused yield."*

In [0]:
def my_decorator(func):
    
    def wrapper():
        print("this is  the before funtion")
        func()
        print("this is  the after funtion")
    return wrapper

@my_decorator
def say_hello():
    print("Hello, world!")
say_hello()
print(say_hello.__name__)

## Decorator — Simple Notes

### What is a decorator?
* A decorator is a function that **takes a function, adds extra behaviour around it, and returns the enhanced version** — without changing the original function's code.
* `@my_decorator` above `say_hello` is just a shortcut for: `say_hello = my_decorator(say_hello)`.

### How does it work?
* `wrapper()` is the new function: it runs code **before** the original ("before function"), calls the original, then runs code **after** ("after function").
* So `say_hello()` no longer prints only "Hello, world!" — it is wrapped inside extra before/after steps.

### Why do we use it?
* To **add features without touching the original code** — logging, timing, authentication, retries are all done with decorators in real projects.

### The problem shown here
* After decorating, `say_hello.__name__` prints **`wrapper`**, not `say_hello` — the decorator **replaced the original function's identity** (name, docstring). That's the bug `functools.wraps` fixes in the next cell.

### Interview one-liner
*"A decorator wraps a function to add behaviour without modifying it — `@dec` is just `func = dec(func)`."*

In [0]:


from  functools import wraps


def my_decorator(func):
    @wraps(func)
    def wrapper():
        print("this is  the before funtion")
        func()
        print("this is  the after funtion")
    return wrapper

@my_decorator
def say_hello():
    print("Hello, world!")
say_hello()
print(say_hello.__name__)

## `functools.wraps` — Simple Notes

### What is `@wraps`?
* A small helper from the `functools` module that you put **inside the decorator, above `wrapper`**.
* It **copies the original function's identity** (name `__name__`, docstring `__doc__`, etc.) onto `wrapper`.

### Why do we need it?
* Without it, the decorated function **loses its identity** — `say_hello.__name__` becomes `wrapper`.
* This breaks real things: debugging tools, logs, documentation, and introspection all show the wrong function name.

### What changed here?
* Same decorator as before, but now `say_hello.__name__` correctly prints **`say_hello`** — identity preserved, behaviour still wrapped.

### Easy way to remember
* Decorator = **wrap the behaviour**; `@wraps` = **keep the identity**.
* Rule of thumb: **every decorator you write should use `@wraps(func)`** — it's the professional standard.

### Interview one-liner
*"`@wraps(func)` preserves the wrapped function's metadata so decorated functions keep their real name and docstring — standard practice in every production decorator."*

In [0]:
from   functools  import   wraps 

def  decorator(func):
    @wraps(func)
    def wraper(*args ,  **Kargs):
        print(f"the  args  are {args} and the  kwargs are {Kargs}" )
        result = func(*args ,  **Kargs)
        print(f"Finished  the  function  {func.__name__} with result {result}" )
        return result
    return wraper

@decorator
def  brew_chai(type ,  milk="no"):
    print(f"brewing  {type}  chai  with  {milk}  milk")


brew_chai("masala" ,  milk="yes")




## Real-Life Decorator (`*args` / `**kwargs`) — Simple Notes

### What is this decorator doing?
* It's a **logging decorator** — it watches any function you put `@decorator` on, and reports what came in and what went out.
* This is the **standard real-world shape** of a decorator — used for logging, timing, authentication in actual projects.

### What is `*args, **kwargs`?
* They make the wrapper **accept any function** — any number of positional and keyword arguments.
* The wrapper collects whatever the caller passes, hands it to the real function, and returns the result.
* So one decorator works on **every** function — not just on functions with fixed arguments.
* *Little detail:* the usual spelling is `**kwargs` (lowercase) — `Kargs` works here, but stick to the standard name in interviews and code reviews.

### Why `result = func(...)` and `return result`?
* A decorator must **return what the real function returns** — otherwise every decorated function would return `None` and break the program.

### The detail in the output
* `Finished the function brew_chai with result None` — `brew_chai` has no `return`, so its result **is** `None`. The decorator honestly reported it; nothing is broken.
* `@wraps(func)` is back — so `brew_chai` keeps its real name and docstring.

### Easy way to remember
* `*args` = catch **all positional** values · `**kwargs` = catch **all keyword** values
* A good decorator = **print before · call the function · print after · return the result**

### Interview one-liner
*"A production decorator accepts `*args, **kwargs` so it can wrap any function, and it must return the wrapped function's result — otherwise it silently breaks return values."*

In [0]:

from  functools import wraps
def require_admin(func):
    @wraps(func)
    def wrapper(user_mode):
        if (user_mode != 'admin'):
            print("Acess not  granted")
            return None
        else:
            result =  func(user_mode)
            return result
    return wrapper

@require_admin
def  acress_tea_inventory(role):
    print("Acess granted to  the invetory")
acress_tea_inventory('admin')
acress_tea_inventory('user')
        
